# IAPT: Benchmarking Small-Object Detection on Brain MRI: A Cerebral Microbleed Case Study

---
---

## Setup

### Libraries

In [ ]:
# installing dependancies
!pip install nnunetv2=2.7.0 nibabel=5.4.2 tqdm scipy -q

# libraries
from google.colab import drive
from pathlib import Path
from tqdm.auto import tqdm
import torch
import os
import logging
import subprocess
import numpy as np
import json
import nibabel as nib
from scipy.ndimage import label
from typing import Literal


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Logger Setup

In [ ]:
############
# initilising and setting up the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s',
    force=True
)
logging.getLogger('nibabel.nifti1').setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

### Configuration Constants

In [ ]:
############
# paths / files / constants

# paths
DRIVE_PATH = Path("/content/drive")

# hardcoded paths as nnUNet take strict paths
BASE_PATH = DRIVE_PATH / "MyDrive/iAPT_nnUNet-Matthias"

TEST_PATH = DRIVE_PATH / "MyDrive/data_sample"
BASE_PATH = TEST_PATH

paths = {
    "nnUNet_raw": BASE_PATH / "nnUNet_raw",
    "nnUNet_preprocessed": BASE_PATH / "nnUNet_preprocessed",
    "nnUNet_results": BASE_PATH / "nnUNet_results"
}

DATASET_PATH = paths["nnUNet_raw"] / "Dataset001_VALDO"

IMAGES_PATH: Path = DATASET_PATH / "imagesTr"
LABELS_PATH: Path = DATASET_PATH / "labelsTr"
STATS_PATH: Path = DATASET_PATH / "verification_stats"
RESULS_OUTPUT_PATH = BASE_PATH / "inference_results"

ALL_PATHS = list(paths.values()).extend([IMAGES_PATH, LABELS_PATH, STATS_PATH, RESULS_OUTPUT_PATH])

# files
METADATA_FILE = "dataset.json"
STATS_FILE = "stats.json"

# constants
MODALITY_SUFFIXES: dict[str, str] = {
    "T1": "0000", 
    "T2": "0001",
    "T2s": "0002"
}

DATA_TYPE: str = ".nii.gz"
K_FOLDS = 5

### Helper Functions

In [ ]:
############
# directory creator helper function
def create_dirs(paths: list[Path] | Path, parents: bool = True, exist_ok: bool = True) -> None:
    if isinstance(paths, Path):
        paths = [paths]

    created = []

    for path in paths:
        if not path.exists():
            created.append(str(path))
        path.mkdir(parents=parents, exist_ok=exist_ok)

    if created:
        logger.info(f"ℹ️ Created {len(created)} directorie(s): {', '.join(created)}")
    else:
        logger.info("ℹ️ All directories already exist")

# file getter helper
PathType = Literal['label', 'image']
def get_files(path_type:PathType) -> list:
    if path_type == 'image':
        path = IMAGES_PATH
    elif path_type == 'label':
        path = LABELS_PATH
    else:
        raise ValueError(f"Invalid value for path_type. Expected: {PathType} Got: {path_type}.")
    
    files = list(path.glob(f"*{DATA_TYPE}"))

    if len(files) == 0:
        logger.error("❌ No label files found.")
        return []
    return files


---

## Google Drive Mounting

Mounting the notebook to the google drive to have access to the data.

In [ ]:
# mounting the notebook to the Google Drive to have access to my drive and shared drives
path_str = str(DRIVE_PATH)
drive.mount(path_str)
logger.info(f"✅ Google Drive mounted to path '{path_str}'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---

## Setting Up Environment Variables

During this section we setup the environment variables as well as creating any necessary directories.

In [ ]:
# creating and logging all directories
create_dirs(ALL_PATHS)

# setting up envars for nnUNet
for key, path in paths.items():
    os.environ[key] = str(path)

logger.info("✅ Created all necessary directories and set nnUNet environment variables.")

INFO - ℹ️ Directories already exist


## Affine Alignment, Dataset Conversion & Metadata

In [ ]:
#logger configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s',
)
nib.imageglobals.logger.setLevel(logging.WARNING) 
logging.getLogger('nibabel').setLevel(logging.WARNING) # supressing warnings that are being fixed in the script
logger = logging.getLogger(__name__)

ROOT_DIR: Path = Path(__file__).resolve().parent.parent
print(ROOT_DIR)
DATA_DIR: Path = ROOT_DIR / "data"

def valdo_to_nnu() -> int:
    logger.info(f"ℹ️ Converting Valdo Dataset to fit nnUNet format")

    subjects = [f for f in ORIG_DATA_PATH.iterdir() if f.is_dir() and f.name.startswith(SUBJECT_PREFIX)] # 72 subjects
    
    sub_count = 0
    failed_sub = []

    for sub_dir in subjects: # iterating over all 72 subjects
        sub_id = sub_dir.name
        id = sub_id.replace(SUBJECT_PREFIX, "")

        # creating mappings (orig name: dest path)
        mappings = {
            # label
            f"{sub_id}_space-T2S_CMB{DATA_TYPE}": LABELS_PATH / f"VALDO_{id}{DATA_TYPE}",
            # images
            f"{sub_id}_space-T2S_desc-masked_T1{DATA_TYPE}": IMAGES_PATH / f"VALDO_{id}_{MODALITY_SUFFIXES[0]}{DATA_TYPE}", # T1
            f"{sub_id}_space-T2S_desc-masked_T2{DATA_TYPE}": IMAGES_PATH / f"VALDO_{id}_{MODALITY_SUFFIXES[1]}{DATA_TYPE}", # T2
            f"{sub_id}_space-T2S_desc-masked_T2S{DATA_TYPE}": IMAGES_PATH / f"VALDO_{id}_{MODALITY_SUFFIXES[2]}{DATA_TYPE}", # T2S
        }

        # moving the files
        mv_count = 0
        for orig_name, target_path in mappings.items():
            file_path = sub_dir / orig_name
            if file_path.exists():
                # saving through nibabel to fix pixdim[0] qfac warning (warning fix)
                img = nib.load(file_path)
                nib.save(img, target_path)
                mv_count += 1
            else:
                logger.warning(f"⚠️ Missing file '{orig_name}' in '{sub_id}'")

        if mv_count == 4:
            sub_count += 1
            logger.debug(f"🔍 Successfully processed '{sub_id}'")
        else:
            logger.error(f"❌ Subject '{sub_id}' is incomplete. Found {mv_count}/4 files.")
            failed_sub.append(sub_id)

    if failed_sub:
        logger.error(f"❌ Failed to process the following subjects: {failed_sub}")

    return sub_count

def add_metadata(dataset_size) -> None:
    metadata = { 
        "channel_names": { 
            "0": "T1", 
            "1": "T2", 
            "2": "T2star" 
        }, 
        "labels": { 
            "background": 0, 
            "microbleed": 1 
        }, 
        "numTraining": dataset_size, 
        "file_ending": ".nii.gz" 
    }

    meta_path = DATASET_PATH / METADATA_FILE
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=4)
    logger.info(f"ℹ️ Added metadata to {str(meta_path)}.")

if __name__ == "__main__":
    logger.info(f"ℹ️ Starting nnUNet dataset setup...")
    # creating target data directory
    create_dirs([IMAGES_PATH, LABELS_PATH])
    # converting source data to target data
    dataset_size = valdo_to_nnu()
    # adding metadata
    add_metadata(dataset_size)
    logger.info("✅ Successfully finished setting up the nnUNet dataset.")

---

## GPU Verification

In [ ]:
# getting nvidia smi output
try:
    nvd_out = subprocess.check_output(["nvidia-smi"]).decode()
    logger.info(f"️️ℹ️ NVIDIA-SMI Output:\n{nvd_out}")
except Exception as e:
    logger.error(f"❌ nvidia-smi not available: {e}")
    raise RuntimeError("GPU not available at runtime. (Fix: Runtime -> Change runtime type -> GPU)")

# check cuda
if torch.cuda.is_available():
    logger.info(f"️️ℹ️ GPU Detected: {torch.cuda.get_device_name(0)}")

    # testing that the GPU is functional
    x = torch.rand(1000, 1000).cuda()
    logger.info(f"️️️️ℹ️ GPU Functioning, tensor running on: {x.device}")
else:
    logger.error("❌ No CUDA GPUs detected.")
    raise RuntimeError("GPU not available at runtime. (Fix: Runtime -> Change runtime type -> GPU)")

INFO - ️️ℹ️ NVIDIA-SMI Output:
Mon Apr  6 12:16:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------

---

## Alignment

In [ ]:
def align_lbls_to_imgs():
    lbl_files = get_files(path_type='label')
    
    logger.info("ℹ️ Aligning label headers to match the images.")
    for lbl in lbl_files:
        sub = lbl.name
        img = IMAGES_PATH / f"{sub}_{MODALITY_SUFFIXES['T2s']}"

        # not major error since we are just doing some preprocessing which will slightly improve the model
        if not img.exists():
            logger.warning(f"⚠️ No matching images found for label subject {sub}.")
            continue

        img_nii = nib.load(img)
        lbl_nii = nib.load(lbl)

        aligned_lbl_nii = nib.Nifti1Image(
            lbl_nii.get_fdata(),
            img_nii.affine,
            img_nii.header
        )

        nib.save(aligned_lbl_nii, lbl) # inplace save

    logger.info("✅ Alignment compleated.")

align_lbls_to_imgs()
    

---

## Label Verification

In [ ]:
class LabelVerification:
    def __init__(self):
        self.corrupt_lbls = []
        self.misaligned_lbls = set()
        self.missing_mod = set()
        self.component_stats = {}

        try:
            with open(DATASET_PATH / METADATA_FILE, 'r') as file:
                data = json.load(file)
                self.dataset_size = data.get('numTraining', 0)
        except Exception as e:
            logger.error(f"❌ Failed to load {METADATA_FILE}: {e}")
            self.dataset_size = 0

    def _lbl_val_integrity(self, sub_id:str, lbl_data) -> bool:
        unique_vals = np.unique(lbl_data)
        if lbl_data.max() > 1 or lbl_data.min() < 0: # checking label mask range is between 0 and 1
            logger.error(f"❌ Label for '{sub_id}' is corrupt.")
            self.corrupt_lbls.append(sub_id)
            return False
        return True

    def _spatial_alignment(self, sub_id:str, lbl_img) -> None:
        for mod in MODALITY_SUFFIXES.values():
            img_path = IMAGES_PATH / f"{sub_id}_{mod}{DATA_TYPE}"

            if not img_path.exists():
                logger.error(f"❌ Missing Modality: '{sub_id}' is missing '{mod}'")
                self.missing_mod.add(f"{sub_id}_{mod}")
                continue

            img = nib.load(img_path)

            # dimension check
            if img.header.get_data_shape() != lbl_img.header.get_data_shape():
                logger.error(f"❌ Dimension Mismatch: '{sub_id}' ({mod})")
                self.misaligned_lbls.add(f"{sub_id}_{mod}")

            # spatial mapping check
            if not np.allclose(img.affine, lbl_img.affine, atol=1e-3):
                logger.error(f"❌ Affine Matrix Mismatch: '{sub_id}' ({mod})")
                self.misaligned_lbls.add(f"{sub_id}_{mod}")

    def _compute_connected_component(self, sub_id: str, lbl_img, lbl_data) -> None:
        # lbl_data == 1 is a microbleed
        label_mask, bleeds_count = label(lbl_data == 1)

        bleed_volumes = []
        voxel_spacing = ()

        # positive cases
        if bleeds_count > 0:
            # calc dimentions of bleed
            voxel_spacing = lbl_img.header.get_zooms()[:3] # ignoring 4th dim (time dim for fmri
            # get vol of each voxel
            voxel_vol = np.prod(voxel_spacing)

            # get size of each bleed (in voxel)
            bleed_sizes = np.bincount(label_mask.ravel())[1:] #skipping background dim (0s)
            # converting the bleed sizes to volumes
            bleed_volumes = (bleed_sizes * voxel_vol).tolist()

        self.component_stats[sub_id] = {
            'microbleed_count': int(bleeds_count),
            'microbleed_vol_mm3': [float(i) for i in bleed_volumes],
            'total_microbleed_vol_mm3': float(np.sum(bleed_volumes)),
            'voxel_spacing': tuple(float(i) for i in voxel_spacing),
            'max_bleed_vol_mm3': float(max(bleed_volumes)) if bleed_volumes else 0.0,
            'min_bleed_vol_mm3': float(min(bleed_volumes)) if bleed_volumes else 0.0,
        }

    def verify_labels(self) -> None:
        lbl_files = get_files('label')
        lbl_size = len(lbl_files)

        # fail if we cannot extract all of the labels
        if lbl_size != self.dataset_size:
            logger.error(f"❌ Extraced only {lbl_size}/{self.dataset_size} labels.")
            return

        logger.info(f"ℹ️ Starting label verification for {self.dataset_size} subjects...")
        for lbl_path in tqdm(lbl_files, desc="Verifying Subjects", unit="subjects"):
            sub_id = lbl_path.name.replace(DATA_TYPE, "")

            try:
                lbl_img = nib.load(lbl_path)
                lbl_data = np.asanyarray(lbl_img.dataobj)
            except Exception as e:
                logger.error(f"❌ Failed to load {sub_id}: {e}")
                self.corrupt_lbls.append(sub_id)
                continue

            # label val integrity
            if not self._lbl_val_integrity(sub_id, lbl_data):
                continue

            # spatial alignment
            self._spatial_alignment(sub_id, lbl_img)

            # connected component statistics
            self._compute_connected_component(sub_id, lbl_img, lbl_data)

        if not self.corrupt_lbls and not self.misaligned_lbls and not self.missing_mod:
            logger.info("✅ All checks passed succesfully")
            return
        logger.warning(f"⚠️ Issues found: {len(self.corrupt_lbls)} corrupt labels, {len(self.misaligned_lbls)} misaligned labels, {len(self.missing_mod)} missing modalities")

    def generate_summary(self) -> None:
        if not self.component_stats:
            logger.warning("⚠️ No statistics detected.")
            return

        stats = self.component_stats
        summary = {
            'total_subjects': len(stats),
            'statistics': stats
        }

        path = STATS_PATH / STATS_FILE
        create_dirs(STATS_PATH)
        with open(path, 'w') as f:
            json.dump(summary, f, indent=4)
        logger.info(f"ℹ️ Statistics saved to {path}")


verifier = LabelVerification()

stat_filepath = STATS_PATH / STATS_FILE
if not (stat_filepath).exists():
    logger.info("ℹ️ No existing stats found. Starting verification...")
    verifier.verify_labels()
    verifier.generate_summary()
else:
    try:
        with open(stat_filepath, 'r') as f:
            data = json.load(f)
            # checking if the stats file is valid
            if data.get('statistics'):
                logger.info(f"✅ Valid stats found for {data.get('total_subjects', 0)} subjects. Skipping.")
            else:
                raise ValueError("Stats file is empty of data.")
    # rerunning verification if the stats file is not valid
    except (json.JSONDecodeError, ValueError):
        logger.warning("⚠️ Stats file was corrupted or empty. Re-running verification...")
        verifier.verify_labels()
        verifier.generate_summary()

INFO - ✅ Valid stats found for 5 subjects. Skipping.


---

## Plan & Preprocessing

In [ ]:
! nnUNetv2_plan_and_preprocess -d 001 --verify_dataset_integrity

Fingerprint extraction...
Dataset001_VALDO
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Direction images: (0.995566581576015, -0.04738368059472497, -0.08125248984004746, -0.0537746879913086, -0.9954736252229431, -0.0783616274266939, 0.07717165658767007, -0.08238354385912565, 0.9936083173171574). 
Direction seg: (0.9955666595640563, -0.047383717925745056, -0.08125151530327597, -0.05377464715622861, -0.9954736255029949, -0.07836165121471604, 0.0771706789375951, -0.08238351900384229, 0.9936083951335277). 
Image files: ['/content/drive/MyDrive/data_sample/nnUNet_raw/Dataset001_VALDO/imagesTr/VALDO_101_0000.nii.gz', '/content/drive/MyDrive/data_sample/nnUNet_raw/Dataset001_VALDO/imagesTr/VALDO_101_0001.nii.gz', '/content/drive/MyDrive/data_sample/nnUNet_raw/Dataset001_VALDO/imagesTr/VALDO_101_0002.nii.gz']. 
Seg file: /content/drive/MyDrive/data_sample/nnUNet_raw/Dataset001_VALDO/labelsTr/VALDO_101.nii.gz

Direction images: (0.9915508391685472, -0.09

---

## Training

In [ ]:
logger.info("ℹ️ Starting training...")
for k in range(K_FOLDS):
    logger.info(f"ℹ️ Training Fold {k}")
    ! nnUNetv2_train 001 3d_fullres {k}

INFO - ℹ️ Starting training...
INFO - ℹ️ Training Fold 0



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-04-06 12:20:19.500678: Using torch.compile...
2026-04-06 12:20:22.225785: do_dummy_2d_data_aug: True
2026-04-06 12:20:22.231211: Creating new 5-fold cross-validation split...
2026-04-06 12:20:22.244171: Desired fold for training: 0
2026-04-06 12:20:22.247833: This split has 4 training and 1

INFO - ℹ️ Training Fold 1


Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_train", line 5, in <module>
    from nnunetv2.run.run_training import run_training_entry
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/run/run_training.py", line 7, in <module>
    import torch.cuda
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 430, in <module>
    _load_global_deps()
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 381, in _load_global_deps
    _preload_cuda_deps()
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 348, in _preload_cuda_deps
    _preload_cuda_lib(lib_folder, lib_name)
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 318, in _preload_cuda_lib
    ctypes.CDLL(lib_path)
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


INFO - ℹ️ Training Fold 2


^C
Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_train", line 5, in <module>
    from nnunetv2.run.run_training import run_training_entry
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/run/run_training.py", line 12, in <module>
    from nnunetv2.run.load_pretrained_weights import load_pretrained_weights
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/run/load_pretrained_weights.py", line 2, in <module>
  File "/usr/local/lib/python3.12/dist-packages/torch/_dynamo/__init__.py", line 76, in <module>
    from .polyfills import loader as _  # usort: skip # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/_dynamo/polyfills/loader.py", line 32, in <module>
    POLYFILLED_MODULES: tuple["ModuleType", ...] = tuple(
                                                   ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/_dynamo/polyfills/loader.py", line 33, in <genexpr>
    importlib.import_mod

INFO - ℹ️ Training Fold 3
INFO - ℹ️ Training Fold 4


^C
^C


---

## Inference

In [ ]:
create_dirs(RESULS_OUTPUT_PATH)
! nnUNetv2_predict -i {str(IMAGES_PATH)} \
                   -o {str(RESULS_OUTPUT_PATH)} \
                   -d 001 \
                   -c 3d_fullres \
                   -f 0 1 2 3 4 --save_probabilities

### Evaluation